# American Stories KWIC controller

这个 notebook 只负责控制数据获取参数并调用 `scripts/americanstories_kwic.py`。下载、解析 archive、抽取 KWIC 的具体逻辑都放在脚本里，避免 notebook 和脚本维护两套重复代码。

请在 Jupyter 里选择 `amstories` kernel。如果没有看到这个 kernel，可以先在终端运行：

```bash
conda activate amstories
python -m ipykernel install --user --name amstories --display-name "Python (amstories)"
```

In [1]:
import csv
import subprocess
import sys
from pathlib import Path

print(sys.executable)
assert "amstories" in sys.executable, "请切换到 amstories kernel 后再继续运行。"

/opt/miniconda3/envs/amstories/bin/python


## 1. 设置抽取参数

改这里就可以控制年份、词表、窗口大小、每个词最多保留多少条 KWIC，以及是否缓存 American Stories 年份 archive。

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

years = [str(year) for year in range(1850, 1861)]
terms = [
    "lawyer",
    "engineer"
]

window = 30
max_rows_per_year_term = 10
max_articles = None
max_articles_per_year = 5000
timeout_seconds = 600
progress_every = 1000
keep_archives = True

cache_dir = PROJECT_ROOT / "data" / "raw" / "americanstories"
out_path = PROJECT_ROOT / "data" / "kwic" / f"americanstories_{'_'.join(years)}_kwic.csv"

cache_dir.mkdir(parents=True, exist_ok=True)
out_path.parent.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, out_path

(PosixPath('/Users/doge/Documents/devaluation'),
 PosixPath('/Users/doge/Documents/devaluation/data/kwic/americanstories_1800_1801_1802_1803_1804_1805_1806_1807_1808_1809_1810_kwic.csv'))

## 2. 调用 KWIC 抽取脚本

这一格使用当前 notebook 的 `amstories` Python 来运行脚本，避免调用到 base 环境。

In [3]:
script = PROJECT_ROOT / "scripts" / "americanstories_kwic.py"

cmd = [
    sys.executable,
    str(script),
    "--years", *years,
    "--terms", *terms,
    "--window", str(window),
    "--max-rows-per-year-term", str(max_rows_per_year_term),
    "--max-articles-per-year", str(max_articles_per_year),
    "--timeout-seconds", str(timeout_seconds),
    "--progress-every", str(progress_every),
    "--cache-dir", str(cache_dir),
    "--out", str(out_path),
]

if keep_archives:
    cmd.append("--keep-archives")
if max_articles is not None:
    cmd.extend(["--max-articles", str(max_articles)])

print(" ".join(cmd))
process = subprocess.Popen(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)

try:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
except KeyboardInterrupt:
    print("\nInterrupted: terminating KWIC subprocess...")
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        print("Subprocess did not exit after terminate(); killing it...")
        process.kill()
        process.wait()
    raise

if return_code != 0:
    raise RuntimeError(f"KWIC script failed with exit code {return_code}")

out_path

/opt/miniconda3/envs/amstories/bin/python /Users/doge/Documents/devaluation/scripts/americanstories_kwic.py --years 1800 1801 1802 1803 1804 1805 1806 1807 1808 1809 1810 --terms lawyer engineer --window 30 --max-rows-per-year-term 10 --max-articles-per-year 5000 --timeout-seconds 600 --progress-every 1000 --cache-dir /Users/doge/Documents/devaluation/data/raw/americanstories --out /Users/doge/Documents/devaluation/data/kwic/americanstories_1800_1801_1802_1803_1804_1805_1806_1807_1808_1809_1810_kwic.csv --keep-archives
Scanning 1800
Progress: 1000 articles scanned, 1 rows written, rows by term: {'lawyer': 0, 'engineer': 1}
Progress: 2000 articles scanned, 1 rows written, rows by term: {'lawyer': 0, 'engineer': 1}
Progress: 3000 articles scanned, 1 rows written, rows by term: {'lawyer': 0, 'engineer': 1}
Progress: 4000 articles scanned, 3 rows written, rows by term: {'lawyer': 1, 'engineer': 2}
Finished 1800; cumulative rows: 3
Scanning 1801
Progress: 5000 articles scanned, 4 rows writt

PosixPath('/Users/doge/Documents/devaluation/data/kwic/americanstories_1800_1801_1802_1803_1804_1805_1806_1807_1808_1809_1810_kwic.csv')

## 3. 快速检查输出

这里不做复杂分析，只确认 CSV 行数、列名和前几条 KWIC 是否正常。后续正式分析可以在 R / `conText` 里继续。

In [5]:
with open(out_path, "r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    preview = []
    counts = {term: 0 for term in terms}
    for row in reader:
        if len(preview) < 5:
            preview.append(row)
        if row["keyword"] in counts:
            counts[row["keyword"]] += 1

print("Output:", out_path)
print("Rows by term:", counts)
preview

Output: /Users/doge/Documents/devaluation/data/kwic/americanstories_1800_1801_1802_1803_1804_1805_1806_1807_1808_1809_1810_kwic.csv
Rows by term: {'lawyer': 80, 'engineer': 24}


[{'year': '1800',
  'date': '1800-08-18',
  'newspaper_name': 'Jenks&#39;s Portland gazette. [volume] (Portland [Me.]) 1799-1802',
  'article_id': '19_1800-08-18_p2_sn83016063_00332895059_1800081801_0440',
  'page': 'p2',
  'edition': '1800081801',
  'headline': 'DUBLIN, JUNE Al.',
  'byline': '',
  'keyword': 'engineer',
  'match': 'engineer',
  'left': 'of being at hand to co operate with our allies on the other Italian COATS , as eventual opportunities may require . It . has been proPoGed by an Auitrian',
  'right': ', TO unite , by means Of canals , the Adriatic lea with the Baltic and to open communica ton by water , from The hereditary dominions Of the Emperor',
  'context': 'of being at hand to co operate with our allies on the other Italian COATS , as eventual opportunities may require . It . has been proPoGed by an Auitrian engineer , TO unite , by means Of canals , the Adriatic lea with the Baltic and to open communica ton by water , from The hereditary dominions Of the Emper